# Interloper calibration for LSS — a toy field (simulation only)

A minimal end-to-end grism scene for studying line-redshift interlopers in a Roman GRS-style LSS sample. We populate one SCA with emission-line galaxies (treated as point sources), disperse orders 0 and 1, add detector noise and a zodiacal floor, and boxcar-extract a 1D spectrum per object — everything the downstream redshift-fitting / completeness–purity analysis needs as input.

**Scope.** This notebook stops at the extracted spectra. The redshift fitting and the completeness/purity metrics are deliberately left for a follow-on: here we build and sanity-check the data, and emit a truth table to fit against.

**The interloper geometry.** Each galaxy carries Hα and [O III] 5007. A single observed line at $\lambda$ is degenerate — it could be Hα at $z=\lambda/6563-1$ or [O III] at $z=\lambda/5007-1$ — and breaking that degeneracy (via the second line or the line ratio — or, in principle, the 0th-order position, though the 0th order is faint at this grism's few-percent throughput) is the whole problem. Wherever both lines fall in the grism band (0.9–2.0 µm), the two-line solution resolves it; where only one line is measured, it does not.

Everything below is parametrized in the first cell.

In [ ]:
import os, sys, time
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

nb_root = Path.cwd()
if not (nb_root / "tutorial_helpers.py").exists():
    nb_root = nb_root.parent
sys.path.insert(0, str(nb_root))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax, jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser, pipeline
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

element = GRISM
print("JAX backend:", jax.default_backend(), "| data:", paths.data_dir())

## Parameters

In [ ]:
# --- scene -----------------------------------------------------------------
N_GAL        = 100                 # number of galaxies
SCA          = 5                   # detector (WFI05; PSF cache hydrated)
POS_LO, POS_HI = -500.0, 4500.0    # SCA x,y draw range (1-indexed; spills off-chip on purpose)
Z_LO, Z_HI   = 1.0, 2.0            # redshift range (uniform)
SEED         = 42

# --- spectra ---------------------------------------------------------------
CONT_ABMAG   = 24.0                # flat-f_nu continuum, AB mag (constant in f_nu => same in any band)
F_HALPHA     = 1.0e-15            # Halpha line flux, erg/s/cm^2
OIII_RATIO   = 0.3                 # [OIII]5007 flux as a fraction of Halpha
LINE_V_FWHM  = 150.0               # intrinsic line FWHM, km/s
HALPHA_REST  = 6562.8             # Angstrom (air)
OIII_REST    = 5006.8             # Angstrom (air), [OIII]5007

# --- instrument / exposure -------------------------------------------------
ORDERS       = ["1", "0"]          # spectral orders to deposit (science + 0th)
ROLLS_DEG    = [0.0, 15.0]         # two exposures; roll = rotate the scene about the SCA centre
EXPTIME      = 190.0               # s
ZODI_RATE    = 1.0                 # e-/s/pixel, uniform background
DET          = 4088                # SCA size (pixels)
CENTER       = (DET / 2, DET / 2)  # roll-rotation centre (~2044, 2044)

# --- extraction / display --------------------------------------------------
APERTURE     = 4                   # boxcar half-width, cross-dispersion pixels
SUBTRACT_BKG = True                # subtract a robust per-pixel background before extracting
SMOOTH_FWHM_A = 0.0                # Å, optional smoothing of extracted spectra (0 = off; show the raw per-bin noise)
N_SHOW       = 6                   # galaxies to plot
WINDOW_A     = 250.0               # Å, half-width of the zoom around each line in the spectrum plot
ZOOM_SIZE    = 1000                # px, side of the zoomed field-image region (auto-centred on the busiest area)

C_KMS  = 2.99792458e5
C_AAPS = 2.99792458e18             # speed of light, Angstrom/s (for f_nu -> f_lambda)
BAND_LO_A, BAND_HI_A = 9000.0, 20000.0
rng = np.random.default_rng(SEED)

## 1. Disperser payloads (orders 0 and 1)

One compiled batched disperser per order, on the production 2 Å grid. The batched path (`pipeline.make_batched_star_fori`) takes a FLAM array per source and applies the per-order sensitivity and bin width internally. We use `th.load_sensitivity` (zero outside the tabulated range) so band edges don't pick up spurious flux.

In [ ]:
wl_um, wl_a, dlam_a = th.wavelength_grid(element)         # 2 Å production grid
wl_j = jnp.asarray(wl_um)
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))

opt, foris = {}, {}
for order in ORDERS:
    opt[order] = omj.make_sca_payload(model, sca=SCA, order=order)
    psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order=order, element=element,
                                            cache_dir=str(paths.psf_cache_dir()), verbose=False)
    sens = jnp.asarray(th.load_sensitivity(SCA, order, wl_a, element=element))
    disp = star_disperser.make_star_disperser(psf, opt[order])
    foris[order] = pipeline.make_batched_star_fori(disp, sens, wl_j, dlam_a)
opt1 = opt["1"]
print("orders ready:", list(foris))

## 2. The galaxy catalog and their spectra

Positions and redshifts are drawn uniformly; the spectrum is a flat-$f_\nu$ continuum (so the AB magnitude is the same in every band, and the F158 normalization is trivial) plus two Gaussian emission lines. Line widths are set by `LINE_V_FWHM` in velocity, so the observed-frame width scales with $\lambda_\mathrm{obs}=\lambda_\mathrm{rest}(1+z)$. For modest velocity widths the grism's own resolution (tens of Å) dominates, so the *observed* line is instrument-limited; the intrinsic width matters mainly in that it should stay resolved on the input wavelength grid (cf. validation/01).

In [ ]:
gx = rng.uniform(POS_LO, POS_HI, N_GAL).astype(np.float32)
gy = rng.uniform(POS_LO, POS_HI, N_GAL).astype(np.float32)
gz = rng.uniform(Z_LO, Z_HI, N_GAL).astype(np.float32)

fnu = 10 ** (-0.4 * (CONT_ABMAG + 48.6))                 # erg/s/cm^2/Hz
cont_flam = fnu * C_AAPS / wl_a ** 2                      # erg/s/cm^2/Angstrom

def gauss_area1(lam_a, lam0, fwhm_a):
    sig = fwhm_a / 2.3548
    return np.exp(-0.5 * ((lam_a - lam0) / sig) ** 2) / (sig * np.sqrt(2 * np.pi))

def galaxy_flam(z):
    ha0, o30 = HALPHA_REST * (1 + z), OIII_REST * (1 + z)
    flam = cont_flam.copy()
    flam += F_HALPHA              * gauss_area1(wl_a, ha0, LINE_V_FWHM / C_KMS * ha0)
    flam += F_HALPHA * OIII_RATIO * gauss_area1(wl_a, o30, LINE_V_FWHM / C_KMS * o30)
    return flam

spectra = np.stack([galaxy_flam(z) for z in gz]).astype(np.float32)   # [N_GAL, N_wl]

# observed line centres, and a crude line S/N for the truth table
ha_obs, o3_obs = HALPHA_REST * (1 + gz), OIII_REST * (1 + gz)
sens1_at = lambda lam_a: th.load_sensitivity(SCA, "1", np.atleast_1d(lam_a))
ha_cts = F_HALPHA * sens1_at(ha_obs) * EXPTIME           # e- in the Halpha line
o3_cts = F_HALPHA * OIII_RATIO * sens1_at(o3_obs) * EXPTIME
npix_res = (2 * APERTURE + 1) * 2                         # ~aperture x instrumental FWHM
sn = lambda cts: cts / np.sqrt(npix_res * ZODI_RATE * EXPTIME + cts)
ha_sn, o3_sn = sn(ha_cts), sn(o3_cts)
print(f"continuum: AB={CONT_ABMAG} -> {cont_flam.mean():.2e} erg/s/cm^2/A (mean over band)")
print(f"Halpha   : {np.median(ha_cts):4.0f} e- (median), approx single-exposure S/N {np.median(ha_sn):.1f}")
print(f"[OIII]   : {np.median(o3_cts):4.0f} e- (median), approx single-exposure S/N {np.median(o3_sn):.1f}")
print("(line fluxes scale with F_HALPHA; [OIII] is 0.3x and the weaker of the pair)")

In [ ]:
# one example spectrum, with the lines marked
ig = int(np.argmax(ha_sn))
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(wl_a, spectra[ig], lw=0.8)
for lam, name, col in [(ha_obs[ig], "Hα", "C3"), (o3_obs[ig], "[OIII]", "C2")]:
    ax.axvline(lam, color=col, ls="--", lw=1); ax.text(lam, ax.get_ylim()[1], f" {name}",
        color=col, va="top", fontsize=9)
ax.set(xlabel="observed wavelength [Å]", ylabel="FLAM [erg/s/cm²/Å]",
       title=f"galaxy {ig}: z={gz[ig]:.3f}  (flat $f_ν$ + Hα + [OIII])", xlim=(BAND_LO_A, BAND_HI_A))
fig.tight_layout()

## 3. Simulate the exposures

For each roll we rotate the scene about the SCA centre (the spec's simplification of a roll — the dispersion axis stays fixed on the detector and the sources move), deposit orders 0 and 1, then form the observed image as $\mathrm{Poisson}\!\big[(\text{model rate}+\text{zodi})\times t_\mathrm{exp}\big]$.

In [ ]:
def rotate(x, y, deg):
    if deg == 0.0:
        return x.copy(), y.copy()
    t = np.deg2rad(deg); cx, cy = CENTER
    dx, dy = x - cx, y - cy
    return (cx + dx * np.cos(t) - dy * np.sin(t)).astype(np.float32), \
           (cy + dx * np.sin(t) + dy * np.cos(t)).astype(np.float32)

def disperse_scene(xs, ys):
    rate = jnp.zeros((DET, DET), jnp.float32)
    for order in ORDERS:
        rate = pipeline.disperse_batched_stars(foris[order], spectra, xs, ys, rate, batch_size=1000)
    return np.asarray(rate)

exposures = {}        # roll -> dict(x, y, rate, obs)
for roll in ROLLS_DEG:
    xs, ys = rotate(gx, gy, roll)
    t = time.time(); rate = disperse_scene(xs, ys)
    expected = (rate + ZODI_RATE) * EXPTIME
    obs = rng.poisson(expected).astype(np.float32)
    exposures[roll] = dict(x=xs, y=ys, rate=rate, obs=obs)
    print(f"roll {roll:4.0f}°: dispersed {N_GAL} galaxies x{len(ORDERS)} orders in "
          f"{time.time()-t:4.1f}s; image total {obs.sum():.3e} e-")

### 3a. A zoomed view (the full frame is too sparse to read)

At full-frame scale each source spans a few pixels in 4088² and vanishes — to the eye and to any downsampling. So we zoom to a `ZOOM_SIZE`×`ZOOM_SIZE` region, auto-centred on the busiest part of the field, and build it up in three steps: the clean model, then the real data, then a single trace.

**1. Noiseless model (zodi-free).** The dispersed traces and their bright line spots, near native resolution; cyan marks any source positions inside the window (most traces enter from sources below it). This is the clean picture of what the disperser produces.

In [ ]:
# zoom to a ZOOM_SIZE region, auto-centred on the busiest part of the field
ref = exposures[ROLLS_DEG[0]]["rate"]
best, ZX0, ZY0 = -1.0, 0, 0
for yy in range(0, DET - ZOOM_SIZE + 1, 200):
    for xx in range(0, DET - ZOOM_SIZE + 1, 200):
        s = float(ref[yy:yy + ZOOM_SIZE, xx:xx + ZOOM_SIZE].sum())
        if s > best:
            best, ZX0, ZY0 = s, xx, yy
zx1, zy1 = ZX0 + ZOOM_SIZE, ZY0 + ZOOM_SIZE

fig, axes = plt.subplots(1, len(ROLLS_DEG), figsize=(6 * len(ROLLS_DEG), 6))
for ax, roll in zip(np.atleast_1d(axes), ROLLS_DEG):
    r = exposures[roll]["rate"][ZY0:zy1, ZX0:zx1]   # zodi-free, noiseless
    vmax = np.percentile(r[r > 0], 99.5) if (r > 0).any() else 1.0
    ax.imshow(r, origin="lower", cmap="inferno", extent=[ZX0, zx1, ZY0, zy1],
              norm=AsinhNorm(linear_width=max(vmax * 0.01, 1e-4), vmin=0, vmax=vmax))
    e = exposures[roll]
    m = (e["x"] >= ZX0) & (e["x"] <= zx1) & (e["y"] >= ZY0) & (e["y"] <= zy1)
    ax.scatter(e["x"][m] - 1, e["y"][m] - 1, s=60, facecolors="none", edgecolors="cyan", lw=0.6)
    ax.set(title=f"model only (zodi-free, {ZOOM_SIZE}px zoom), roll {roll:.0f}°",
           xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

**2. Observed (background-subtracted).** The same window in the actual data. The zodi *shot noise* (~$\sqrt{\text{zodi}\times t_\mathrm{exp}}$ per pixel) buries the faint continuum traces, but the bright 1st-order **line spots** — tens of σ per pixel — still pop above it. Each spot sits *along* its trace, not at the source position. The 0th order is separately faint (a few percent throughput, a handful of electrons per object, below the noise) — negligible here as signal or contaminant.

In [ ]:
fig, axes = plt.subplots(1, len(ROLLS_DEG), figsize=(6 * len(ROLLS_DEG), 6))
for ax, roll in zip(np.atleast_1d(axes), ROLLS_DEG):
    obs = exposures[roll]["obs"]; noise = np.sqrt(np.median(obs))
    res = (obs - np.median(obs))[ZY0:zy1, ZX0:zx1]
    ax.imshow(res, origin="lower", cmap="inferno", extent=[ZX0, zx1, ZY0, zy1],
              norm=AsinhNorm(linear_width=max(noise, 1.0), vmin=2 * noise,
                             vmax=np.percentile(res, 99.9)))
    ax.set(title=f"observed SCA{SCA} (bkg-sub, {ZOOM_SIZE}px zoom), roll {roll:.0f}°",
           xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

**3. A single trace, model vs observed.** Zoomed further, to native resolution: the line spot stands out at tens of σ even in the noise, while the faint continuum trace is lost to it.

In [ ]:
# zoom on one central galaxy's 1st-order trace: model (zodi-free) vs the observed
# (zodi + Poisson) data, native resolution. The line spot survives the noise; the
# faint continuum trace does not.
gc = int(np.argmin((gx - CENTER[0]) ** 2 + (gy - CENTER[1]) ** 2))
tx, ty = th.spectral_trace(opt1, gx[gc], gy[gc], wl_um)
ix, iy = tx - 1, ty - 1                          # 1-indexed trace -> 0-indexed array
x0, x1 = int(max(ix.min() - 25, 0)), int(min(ix.max() + 25, DET))
y0, y1 = int(max(iy.min() - 25, 0)), int(min(iy.max() + 25, DET))
obs0 = exposures[0.0]["obs"]; noise = np.sqrt(np.median(obs0))
model_cut = exposures[0.0]["rate"][y0:y1, x0:x1]
obs_cut = (obs0 - np.median(obs0))[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(8, 7))
panels = [(axes[0], model_cut, "model (zodi-free)",
           max(np.percentile(model_cut[model_cut > 0], 99.8) * 0.01, 1e-4)),
          (axes[1], obs_cut, "observed (zodi + Poisson)", max(noise, 1.0))]
for ax, im, ttl, lw in panels:
    ax.imshow(im, origin="lower", cmap="inferno", extent=[x0, x1, y0, y1],
              norm=AsinhNorm(linear_width=lw, vmin=0, vmax=np.percentile(im, 99.8)))
    for lam, name, col in [(ha_obs[gc], "Hα", "cyan"), (o3_obs[gc], "[OIII]", "lime")]:
        j = int(np.argmin(np.abs(wl_a - lam)))
        ax.plot(ix[j], iy[j], "+", color=col, ms=11, mew=1.5)
    ax.set(title=ttl, xlabel="x [pix]")
axes[0].set_ylabel("y [pix]")
fig.suptitle(f"galaxy {gc} (z={gz[gc]:.2f}): 1st-order trace — model vs observed")
fig.tight_layout()

## 4. Boxcar extraction

A compiled-once dispersion (autodiff $dy/d\lambda$) drives a plain boxcar: at each wavelength, step to the trace pixel, sum $\pm$`APERTURE` cross-dispersion pixels, and scale to counts per wavelength bin. We subtract a robust per-pixel background (the image median, $\approx$ zodi$\times t_\mathrm{exp}$) first, so the extracted continuum sits near zero. Extraction uses each object's *true* position — i.e. perfect astrometry; a real pipeline would centroid a direct image.

In [ ]:
# dispersion dy/dlambda (pix/um), compiled once and reused for every object
def _trace_y(wl, x, y):
    w = jnp.atleast_1d(wl)
    xf, yf = omj.sca_to_fpa(opt1, x, y)
    _, ty = omj.mpa_to_sca(opt1, *omj.trace_beam(opt1, jnp.broadcast_to(xf, w.shape),
                                                 jnp.broadcast_to(yf, w.shape), w))
    return ty[0]
_dydl = jax.jit(jax.vmap(jax.grad(_trace_y, argnums=0), in_axes=(0, None, None)))
_dlam_um = float(np.diff(wl_um).mean())

def extract(image, x, y, aperture=APERTURE):
    tx, ty = th.spectral_trace(opt1, x, y, wl_um)
    dydl = np.asarray(_dydl(wl_j, float(x), float(y)))
    raw = np.zeros(len(wl_um))
    H, W = image.shape
    for i in range(len(wl_um)):
        ix, iy = int(round(float(tx[i]))) - 1, int(round(float(ty[i]))) - 1
        if 0 <= ix < W and 0 <= iy < H:
            raw[i] = image[iy, max(0, ix - aperture):ix + aperture + 1].sum()
    return raw * np.abs(dydl) * _dlam_um

t = time.time()
for roll in ROLLS_DEG:
    e = exposures[roll]
    img = e["obs"] - (np.median(e["obs"]) if SUBTRACT_BKG else 0.0)
    e["spec"] = np.stack([extract(img, e["x"][g], e["y"][g]) for g in range(N_GAL)])
print(f"extracted {N_GAL} x {len(ROLLS_DEG)} spectra in {time.time()-t:.1f}s")

## 5. Truth table

What the deferred redshift-fitting step needs: per galaxy, the true redshift, the observed line wavelengths, whether the order-1 trace lands on the detector, and a rough line S/N. The on-chip fraction quantifies the loss from drawing positions that spill past the SCA edges.

In [ ]:
def trace_onchip_frac(x, y):
    tx, ty = th.spectral_trace(opt1, x, y, wl_um)
    on = (tx >= 1) & (tx <= DET) & (ty >= 1) & (ty <= DET)
    return float(on.mean())

onchip = np.array([trace_onchip_frac(gx[g], gy[g]) for g in range(N_GAL)])
truth = dict(id=np.arange(N_GAL), x=gx, y=gy, z=gz,
             halpha_obs=ha_obs, oiii_obs=o3_obs,
             onchip_frac=onchip, halpha_SN=ha_sn)

print(f"{'id':>3} {'x':>7} {'y':>7} {'z':>5} {'Hα_obs':>8} {'[OIII]_obs':>10} "
      f"{'onchip':>7} {'Hα S/N':>7}")
order_sn = np.argsort(-ha_sn)
for g in order_sn[:12]:
    print(f"{g:3d} {gx[g]:7.0f} {gy[g]:7.0f} {gz[g]:5.2f} {ha_obs[g]:8.0f} "
          f"{o3_obs[g]:10.0f} {onchip[g]:7.2f} {ha_sn[g]:7.1f}")
print(f"\n{(onchip > 0).sum()}/{N_GAL} galaxies have some order-1 trace on-chip; "
      f"{(onchip > 0.5).sum()} have >50% on-chip.")

## 6. Extracted spectra with the true line positions

The `N_SHOW` highest-S/N galaxies (on-chip traces), zoomed on each line: left Hα, right [O III]. Each panel overlays the rolls (thin grey) and their mean (black); the spectra are shown **raw — no smoothing** — so the per-bin noise is visible (set `SMOOTH_FWHM_A` > 0 to smooth). The dashed line marks the *true* observed wavelength, and the title gives the mean spectrum's peak in units of its line-free σ.

Two things to read off. First, when both lines are detected the line *pair* pins the redshift unambiguously; the interloper problem lives in the single-line cases — a galaxy whose trace runs off the SCA edge so only one of its two lines lands on-chip, a line buried under a neighbour's trace or 0th-order blob, or a line too faint to detect. Anywhere a *single* observed line is all you have, the Hα vs [O III] redshift degeneracy reopens. Which of these dominates depends on the flux, depth, and crowding you set above. Second, the **roll mean** beats the individual rolls: contamination (a neighbour's overlapping 1st-order trace) moves with the scene between rolls and partly averages down, while a real line sits at the same wavelength in both — a concrete reason the survey takes multiple rolls.

In [ ]:
def smooth(y):
    # Gaussian smoothing to ~a resolution element, so the S/N~few lines show
    # against the per-bin boxcar noise (the line spans ~a resolution element).
    if SMOOTH_FWHM_A <= 0:
        return y
    sig = SMOOTH_FWHM_A / 2.3548 / dlam_a
    n = int(np.ceil(4 * sig))
    k = np.exp(-0.5 * (np.arange(-n, n + 1) / sig) ** 2); k /= k.sum()
    return np.convolve(y, k, mode="same")

pick = [g for g in order_sn if onchip[g] > 0.5][:N_SHOW]
# zoom on each line; overlay the rolls (thin) and their mean (thick). Combining
# rolls suppresses contamination (which moves with the scene) while the real line
# stays put, so the mean is the cleaner detector of a line at a fixed wavelength.
fig, axes = plt.subplots(len(pick), 2, figsize=(12, 2.2 * len(pick)), squeeze=False)
for row, g in enumerate(pick):
    specs = {roll: smooth(exposures[roll]["spec"][g]) for roll in ROLLS_DEG}
    mean = np.mean([specs[r] for r in ROLLS_DEG], axis=0)
    free = (np.abs(wl_a - ha_obs[g]) > 120) & (np.abs(wl_a - o3_obs[g]) > 120) \
           & (wl_a > 10000) & (wl_a < 19000)
    nstd = np.std(mean[free]) or 1.0
    for col, (l0, name, lc) in enumerate([(ha_obs[g], "Hα", "C3"), (o3_obs[g], "[OIII]", "C2")]):
        ax = axes[row, col]
        for roll, c in zip(ROLLS_DEG, ["0.75", "0.55"]):
            ax.plot(wl_a, specs[roll], lw=0.6, color=c, label=f"roll {roll:.0f}°")
        ax.plot(wl_a, mean, lw=1.3, color="k", label="roll mean")
        ax.axvline(l0, color=lc, ls="--", lw=1)
        peak = mean[np.abs(wl_a - l0) < 40].max()
        ax.set(xlim=(l0 - WINDOW_A, l0 + WINDOW_A),
               title=f"g{g}  z={gz[g]:.2f} — {name} (peak {peak/nstd:.1f}σ)")
        ax.title.set_size(9); ax.tick_params(labelsize=7)
axes[0, 0].legend(fontsize=7, loc="upper left")
for ax in axes[-1]:
    ax.set_xlabel("observed wavelength [Å]")
for ax in axes[:, 0]:
    ax.set_ylabel("extracted e⁻/bin")
fig.tight_layout()

## 7. What's set up, and what's deferred

We have, for one SCA at each roll: a noisy grism image (orders 0+1, zodi + Poisson), a background-subtracted boxcar spectrum per galaxy, and a truth table (redshift, observed line positions, on-chip flag, line S/N). That is the input the interloper study consumes.

**Deferred (next notebook):** redshift fitting on the extracted spectra — e.g. match a single detected line against the Hα and [O III] hypotheses, use the second line / line ratio / 0th-order position to break the degeneracy — and the completeness and purity metrics as a function of magnitude, redshift, line S/N, and crowding.

**Caveats to carry forward, by design of this toy.**
- *Faint continuum.* For the magnitude and exposure set above the flat-$f_\nu$ continuum is near the per-pixel noise, so these are effectively line-only spectra — realistic for ELG selection, but the continuum carries little weight.
- *Off-chip losses.* Positions are drawn past the SCA edges, so a fraction of objects have no usable trace (see §5); their spectra are empty or truncated.
- *Faint 0th order.* The 0th-order throughput is only a few percent of the 1st, so it deposits a handful of electrons per object — negligible here as signal or contaminant, despite being one of the orders we disperse.
- *Order 2 omitted.* Per the spec only 0th and 1st orders are deposited; 2nd-order overlap is itself an interloper-like contaminant and is not modelled here.
- *Perfect astrometry.* Extraction uses true positions; a real pipeline centroids a direct image, adding trace-placement error.
- *Single line per species.* No [N II] flanking Hα, no [O III] 4959, no Hβ — add these to harden the redshift-fitting test.

*(Reproducibility: SEED fixed; disperser pinned in `pixi.toml`; reference data version in `data/data-versions.lock`. Rest wavelengths are air values.)*